# STRATA demo

A minimal, self-contained demo of the STRATA model: build it, run a
forward pass on a synthetic batch, and (optionally) load a trained
checkpoint and run inference on one real NPZ patch.

This notebook is for quick interactive checks -- for actual training and
evaluation, use the scripts under `scripts/`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
from src.model import STRATA

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Build the model and check output shape

In [ ]:
model = STRATA(num_classes=4).to(device)
model.eval()

dummy_image = torch.randn(2, 3, 256, 256, device=device)
dummy_building = torch.randint(0, 2, (2, 256, 256), device=device).float()

with torch.no_grad():
    logits = model(dummy_image, building_mask=dummy_building)

print("logits shape:", logits.shape)  # expected: [2, 4, 256, 256]

## 2. (Optional) Load a trained checkpoint and run inference on one NPZ patch

Set `CHECKPOINT_PATH` and `SAMPLE_NPZ_PATH` to real paths to try this.

In [ ]:
CHECKPOINT_PATH = None  # e.g. "runs/main/optical_to_optical/STRATA_O2O_n100/models/best_model.pth"
SAMPLE_NPZ_PATH = None  # e.g. "/path/to/ebd_npz/EARTHQUAKE-TURKEY_000001_0_0.npz"

if CHECKPOINT_PATH and SAMPLE_NPZ_PATH:
    import numpy as np
    from src.utils import load_checkpoint, load_model_weights
    from src.datasets import normalize_image, ensure_channel

    checkpoint = load_checkpoint(CHECKPOINT_PATH, map_location=device)
    load_model_weights(model, checkpoint, strict=True)
    model.eval()

    with np.load(SAMPLE_NPZ_PATH) as data:
        img = normalize_image(ensure_channel(data["image"]))
        building = data["building_mask"]
        damage_gt = data["damage_mask"]

    img_t = torch.from_numpy(img.transpose(2, 0, 1)).float().unsqueeze(0).to(device)
    building_t = torch.from_numpy(building.astype("float32")).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = torch.argmax(model(img_t, building_mask=building_t), dim=1)[0].cpu().numpy()

    print("Predicted class counts:", {c: int((pred == c).sum()) for c in range(4)})
    print("Ground-truth class counts:", {c: int((damage_gt == c).sum()) for c in range(4)})
else:
    print("Set CHECKPOINT_PATH and SAMPLE_NPZ_PATH above to try real inference.")